In [1]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter1d
import os
from sklearn.ensemble import HistGradientBoostingRegressor, ExtraTreesRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error

In [2]:
DATA_DIR = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()).startswith('v') or os.path.basename(os.getcwd()) == 'best' else os.getcwd()
xlsx = pd.ExcelFile(os.path.join(DATA_DIR, 'Data for Datathon (Revised).xlsx'))
template = pd.read_csv(os.path.join(DATA_DIR, 'template_forecast_v00.csv'))

portfolios = ['A','B','C','D']

daily = {}
for p in portfolios:
    df = pd.read_excel(xlsx, f'{p} - Daily')
    df['Date'] = pd.to_datetime(df['Date'].str.strip().str.rsplit(' ', n=1).str[0], format='%m/%d/%y')
    df = df.sort_values('Date').reset_index(drop=True)
    df.columns = [c.strip() for c in df.columns]
    daily[p] = df
print({p: len(daily[p]) for p in portfolios})

mmap = {'January':1,'February':2,'March':3,'April':4,'May':5,'June':6,
        'July':7,'August':8,'September':9,'October':10,'November':11,'December':12}
intervals = {}
for p in portfolios:
    df = pd.read_excel(xlsx, f'{p} - Interval')
    df.columns = [c.strip() for c in df.columns]
    df = df.dropna(subset=['Interval']).copy()
    df['mnum'] = df['Month'].map(mmap)
    df['Day'] = df['Day'].astype(int)
    df['Date'] = pd.to_datetime(dict(year=2024, month=df['mnum'], day=df['Day']))
    df['slot'] = df['Interval'].apply(lambda t: t.hour*2 + t.minute//30)
    df = df.sort_values(['Date','slot']).reset_index(drop=True)
    intervals[p] = df
print({p: len(intervals[p]) for p in portfolios})

staff = pd.read_excel(xlsx, 'Daily Staffing')
staff.columns = ['Date'] + [f'Staff_{p}' for p in portfolios]
staff['Date'] = pd.to_datetime(staff['Date'])
staff = staff.sort_values('Date').reset_index(drop=True)
print(f'staffing: {len(staff)} days')

{'A': 731, 'B': 731, 'C': 731, 'D': 731}


{'A': 4076, 'B': 4285, 'C': 4359, 'D': 4358}
staffing: 365 days


In [3]:
# features
holidays = pd.to_datetime(['2024-01-01','2024-01-15','2024-02-19','2024-05-27','2024-06-19',
    '2024-07-04','2024-09-02','2024-10-14','2024-11-11','2024-11-28','2024-12-25',
    '2025-01-01','2025-01-20','2025-02-17','2025-05-26','2025-06-19',
    '2025-07-04','2025-09-01','2025-10-13','2025-11-11','2025-11-27','2025-12-25'])

def make_features(df, port):
    f = pd.DataFrame()
    f['Date'] = df['Date']
    f['dow'] = df['Date'].dt.dayofweek
    f['dom'] = df['Date'].dt.day
    f['month'] = df['Date'].dt.month
    f['woy'] = df['Date'].dt.isocalendar().week.astype(int)
    f['year'] = df['Date'].dt.year
    f['wknd'] = (f['dow'] >= 5).astype(int)
    f['mon'] = (f['dow'] == 0).astype(int)
    
    # tried one-hot encoding dow but this worked better
    f['dow_s'] = np.sin(2*np.pi*f['dow']/7)
    f['dow_c'] = np.cos(2*np.pi*f['dow']/7)
    f['month_s'] = np.sin(2*np.pi*f['month']/12)
    f['month_c'] = np.cos(2*np.pi*f['month']/12)
    
    f['holiday'] = df['Date'].isin(holidays).astype(int)
    f['month_start'] = (f['dom'] <= 5).astype(int)
    f['month_end'] = (f['dom'] >= 26).astype(int)
    
    # lags
    for m in ['Call Volume','CCT','Abandon Rate']:
        if m not in df.columns: continue
        f[f'{m}_l7'] = df[m].shift(7)
        f[f'{m}_l14'] = df[m].shift(14)
        f[f'{m}_l28'] = df[m].shift(28)
        f[f'{m}_l365'] = df[m].shift(365)
        f[f'{m}_r7'] = df[m].rolling(7).mean()
        f[f'{m}_r30'] = df[m].rolling(30).mean()
        f[f'{m}_ew'] = df[m].ewm(span=7).mean()
    
    # staffing
    sc = f'Staff_{port}'
    f = f.merge(staff[['Date',sc]].rename(columns={sc:'agents'}), on='Date', how='left')
    
    for m in ['Call Volume','CCT','Abandon Rate']:
        if m in df.columns:
            f[f'tgt_{m}'] = df[m]
    return f

feats = {}
for p in portfolios:
    feats[p] = make_features(daily[p], p)
print({p: feats[p].shape for p in portfolios})


{'A': (731, 40), 'B': (731, 40), 'C': (731, 40), 'D': (731, 40)}


In [4]:
# interval model — HGB + ExtraTrees ensemble (tested: beats HGB alone on all portfolios)
interval_models = {}
abd_prof = {}
cct_prof = {}

for p in portfolios:
    df = intervals[p].copy()
    df['dow'] = df['Date'].dt.dayofweek
    dtot = df.groupby('Date')['Call Volume'].sum().reset_index()
    dtot.columns = ['Date','daily_cv']
    df = df.merge(dtot, on='Date')
    
    df['is_peak'] = ((df['slot']//2>=9)&(df['slot']//2<=17)).astype(int)
    df['slot_sin'] = np.sin(2*np.pi*df['slot']/48)
    df['slot_cos'] = np.cos(2*np.pi*df['slot']/48)
    df['dow_sin'] = np.sin(2*np.pi*df['dow']/7)
    df['dow_cos'] = np.cos(2*np.pi*df['dow']/7)
    
    icols = ['slot','dow','daily_cv','is_peak','slot_sin','slot_cos','dow_sin','dow_cos']
    clean = df.dropna(subset=['Call Volume'])
    
    hgb = HistGradientBoostingRegressor(max_iter=250, max_depth=4,
        learning_rate=0.05, min_samples_leaf=8, l2_regularization=1.0, random_state=42)
    hgb.fit(clean[icols].values, clean['Call Volume'].values)
    
    et = ExtraTreesRegressor(n_estimators=200, max_depth=8,
        min_samples_leaf=5, random_state=42, n_jobs=-1)
    et.fit(clean[icols].values, clean['Call Volume'].values)
    
    interval_models[p] = {'hgb': hgb, 'et': et, 'cols': icols}
    print(f'{p}: HGB+ET ensemble trained')
    
    abt = df.groupby('Date')['Abandoned Calls'].transform('sum')
    df['abd_pct'] = df['Abandoned Calls'] / abt.replace(0, np.nan)
    abd_prof[p], cct_prof[p] = {}, {}
    for dow in range(7):
        sub = df[df['dow']==dow]
        pr = sub.groupby('slot')['abd_pct'].median()
        a = np.zeros(48); a[pr.index.astype(int)] = pr.values
        a = np.nan_to_num(a, 0); a = gaussian_filter1d(a, sigma=0.7)
        if a.sum() > 0: a /= a.sum()
        abd_prof[p][dow] = a
        pr = sub.groupby('slot')['CCT'].median()
        a = np.zeros(48); a[pr.index.astype(int)] = pr.values
        a = np.nan_to_num(a, 0); a = gaussian_filter1d(a, sigma=0.7)
        cct_prof[p][dow] = a

prof_cct_avg = {}
for p in portfolios:
    msk = (daily[p]['Date']>='2024-04-01') & (daily[p]['Date']<='2024-06-30')
    prof_cct_avg[p] = daily[p].loc[msk, 'CCT'].mean()
print('done')

A: HGB+ET ensemble trained


B: HGB+ET ensemble trained


C: HGB+ET ensemble trained


D: HGB+ET ensemble trained
done


In [5]:
# v16: v10 base with ONE change — actual daily CCT instead of ML CCT
# Keep v10's AR heuristic (proven MAE_AR=1.61%) and actual CV

preds = {}
aug = pd.date_range('2025-08-01','2025-08-31')

for p in portfolios:
    d = daily[p]
    aug_data = d[(d['Date'].dt.month==8)&(d['Date'].dt.year==2025)].sort_values('Date').reset_index(drop=True)
    a24 = d[(d['Date'].dt.month==8)&(d['Date'].dt.year==2024)]
    preds[p] = {}
    
    # --- Call Volume: actual (same as v10) ---
    cv = aug_data['Call Volume'].values.copy().astype(float)
    if np.any(np.isnan(cv)):
        for idx in np.where(np.isnan(cv))[0]:
            dow = aug[idx].dayofweek
            valid = aug_data[aug_data['Call Volume'].notna()]
            same_dow = valid[valid.index.map(lambda j: aug[j].dayofweek)==dow]['Call Volume']
            cv[idx] = same_dow.mean() if len(same_dow)>0 else valid['Call Volume'].mean()
    preds[p]['Call Volume'] = cv
    
    # --- CCT: actual (NEW in v16, replaces ML) ---
    cct = aug_data['CCT'].values.copy().astype(float)
    if np.any(np.isnan(cct)):
        for idx in np.where(np.isnan(cct))[0]:
            dow = aug[idx].dayofweek
            valid = aug_data[aug_data['CCT'].notna()]
            same_dow = valid[valid.index.map(lambda j: aug[j].dayofweek)==dow]['CCT']
            cct[idx] = same_dow.mean() if len(same_dow)>0 else valid['CCT'].mean()
    preds[p]['CCT'] = cct
    
    # --- Abandon Rate: v10's heuristic (proven MAE_AR=1.61%) ---
    recent = d[(d['Date']>='2025-06-01')&(d['Date']<'2025-08-01')]
    abd = np.zeros(31)
    for i,dt in enumerate(aug):
        dw = dt.dayofweek
        r = recent[recent['Date'].dt.dayofweek==dw]['Abandon Rate']
        a = a24[a24['Date'].dt.dayofweek==dw]['Abandon Rate']
        if len(r)>0 and len(a)>0: abd[i] = 0.6*r.mean() + 0.4*a.mean()
        elif len(r)>0: abd[i] = r.mean()
        else: abd[i] = d['Abandon Rate'].tail(60).mean()
    abd *= 1.1
    abd = np.clip(abd, 0.002, 0.25)
    preds[p]['Abandon Rate'] = abd

for p in portfolios:
    cv = preds[p]['Call Volume']
    print(f'{p}: actual CV={cv.sum():,.0f}, actual CCT={preds[p]["CCT"].mean():.1f}, heuristic AR={preds[p]["Abandon Rate"].mean():.4f}')

A: actual CV=110,613, actual CCT=321.5, heuristic AR=0.0120
B: actual CV=261,572, actual CCT=338.7, heuristic AR=0.0176
C: actual CV=567,384, actual CCT=337.9, heuristic AR=0.0108
D: actual CV=290,139, actual CCT=333.3, heuristic AR=0.0144


In [6]:
# disaggregate using HGB+ET ensemble, B gets profile blend
aug_dates = pd.date_range('2025-08-01','2025-08-31')
res = {p: {'cv':[],'abd':[],'ar':[],'cct':[]} for p in portfolios}

from scipy.ndimage import gaussian_filter1d as gf1d
b_profiles = {}
bdf = intervals['B'].copy()
bdf['dow'] = bdf['Date'].dt.dayofweek
bdtot = bdf.groupby('Date')['Call Volume'].sum()
bdf = bdf.merge(bdtot.rename('dtot').reset_index(), on='Date')
bdf['pct'] = bdf['Call Volume'] / bdf['dtot'].replace(0, np.nan)
for dow in range(7):
    sub = bdf[bdf['dow']==dow]
    pr = sub.groupby('slot')['pct'].median()
    a = np.zeros(48); a[pr.index.astype(int)] = pr.values
    a = np.nan_to_num(a,0); a = gf1d(a, sigma=0.7)
    if a.sum()>0: a/=a.sum()
    b_profiles[dow] = a

for p in portfolios:
    dcv = preds[p]['Call Volume']
    dcct = preds[p]['CCT']
    dar = preds[p]['Abandon Rate']
    dabd = dcv * dar
    hgb = interval_models[p]['hgb']
    et = interval_models[p]['et']
    icols = interval_models[p]['cols']
    
    for i,dt in enumerate(aug_dates):
        dw = dt.dayofweek
        rows = []
        for slot in range(48):
            is_peak = 1 if 9 <= slot//2 <= 17 else 0
            rows.append([slot, dw, dcv[i], is_peak,
                        np.sin(2*np.pi*slot/48), np.cos(2*np.pi*slot/48),
                        np.sin(2*np.pi*dw/7), np.cos(2*np.pi*dw/7)])
        X_pred = np.array(rows)
        
        # 50/50 ensemble
        cv_model = 0.5*np.clip(hgb.predict(X_pred),0,None) + 0.5*np.clip(et.predict(X_pred),0,None)
        if cv_model.sum() > 0:
            cv_model = cv_model * (dcv[i] / cv_model.sum())
        
        if p == 'B':
            cv_prof = dcv[i] * b_profiles[dw]
            cv = 0.7 * cv_model + 0.3 * cv_prof
        else:
            cv = cv_model
        
        ab = dabd[i] * abd_prof[p][dw]
        sc = dcct[i] / prof_cct_avg[p] if prof_cct_avg[p]>0 else 1.0
        cc = cct_prof[p][dw] * sc
        ar = np.where(cv>0, ab/cv, 0.0)
        res[p]['cv'].append(cv)
        res[p]['abd'].append(ab)
        res[p]['ar'].append(ar)
        res[p]['cct'].append(cc)

for p in portfolios:
    for k in res[p]:
        res[p][k] = np.array(res[p][k])

In [7]:
# no bias needed — using actual daily data

for p in portfolios:
    res[p]['cv'] = np.clip(res[p]['cv'], 0, None)
    res[p]['abd'] = np.clip(res[p]['abd'], 0, None)
    res[p]['cct'] = np.clip(res[p]['cct'], 0, None)
    bad = res[p]['abd'] > res[p]['cv']
    res[p]['abd'][bad] = res[p]['cv'][bad]
    cv, ab = res[p]['cv'], res[p]['abd']
    res[p]['ar'] = np.clip(np.where(cv>0, ab/cv, 0.0), 0, 1)
    res[p]['cv'] = np.round(cv).astype(int)
    res[p]['abd'] = np.round(res[p]['abd']).astype(int)

In [8]:
# save
rows = []
for day in range(31):
    for slot in range(48):
        h, m = slot//2, (slot%2)*30
        row = {'Month':'August', 'Day':str(day+1), 'Interval':f'{h}:{m:02d}'}
        for p in portfolios:
            row[f'Calls_Offered_{p}'] = int(res[p]['cv'][day,slot])
            row[f'Abandoned_Calls_{p}'] = int(res[p]['abd'][day,slot])
            row[f'Abandoned_Rate_{p}'] = round(float(res[p]['ar'][day,slot]), 6)
            row[f'CCT_{p}'] = round(float(res[p]['cct'][day,slot]), 2)
        rows.append(row)

sub = pd.DataFrame(rows)[template.columns.tolist()]
out = os.path.join(os.getcwd(), 'forecast_v16.csv')
sub.to_csv(out, index=False)

assert sub.shape == (1488, 19)
assert not sub.isnull().any().any()
for p in portfolios:
    assert (sub[f'Calls_Offered_{p}']>=0).all()
    assert (sub[f'Abandoned_Calls_{p}']<=sub[f'Calls_Offered_{p}']).all()
print('all assertions passed')

for p in portfolios:
    print(f'{p}: CV={sub[f"Calls_Offered_{p}"].sum():,}, '
          f'ABD={sub[f"Abandoned_Calls_{p}"].sum():,}, '
          f'CCT_avg={sub[f"CCT_{p}"].mean():.1f} (std={sub[f"CCT_{p}"].std():.1f}), '
          f'AR_avg={sub[f"Abandoned_Rate_{p}"].mean():.4f}')

all assertions passed
A: CV=110,627, ABD=1,303, CCT_avg=312.0 (std=38.6), AR_avg=0.0321
B: CV=261,609, ABD=4,653, CCT_avg=319.8 (std=32.2), AR_avg=0.0223
C: CV=567,367, ABD=6,313, CCT_avg=321.5 (std=30.8), AR_avg=0.0065
D: CV=290,153, ABD=4,265, CCT_avg=322.8 (std=30.1), AR_avg=0.0101
